# Latihan Percobaan Kumar (Trial LoRA) — Qwen2.5-Coder-0.5B

**Ini hanya latihan percobaan, bukan Kumar yang pintar.**
Dataset kita sekarang baru **37 contoh** (`dataset.jsonl`). Dengan data sekecil ini, model hasil latihan akan **menghafal contohnya, bukan memahami ilmunya**. Tujuan latihan ini cuma satu: **membuktikan jalurnya jalan** — GPU nyala, data kebaca, latihan selesai, adapter tersimpan, dan model bisa menjawab.

Yang kamu butuhkan:
- Akun Google dan Google Colab (gratis). Buka notebook ini di Colab.
- GPU gratis: di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Run all**.
- File `dataset.jsonl` dari repo `lieon12/lion-cyber-agent` (folder `dataset/`).

Langkahnya:
1. Jalankan sel **Cek GPU** — harus ada GPU, kalau tidak ada latihan tidak bisa jalan.
2. Jalankan sel **Install** (sekali saja).
3. Saat diminta, **upload `dataset.jsonl`**.
4. Jalankan semua sel sampai selesai. Latihannya cepat (37 contoh, 3 epoch, beberapa menit di T4).
5. Baca hasil ujinya di sel **Uji adapter** — 2 prompt dari data latih (harusnya dihafal) dan 1 prompt baru (tes sebenarnya).

Kalau semua jalan, jalur latihan kita terbukti. Kumar yang benar-benar pintar baru datang setelah dataset tumbuh ke 500+ contoh, lalu latih ulang dengan data itu.

## 1. Cek GPU — latihan ini WAJIB GPU. Tanpa GPU, berhenti di sini.

In [ ]:
import sys

print("== nvidia-smi ==")
!nvidia-smi --query-gpu=name,memory.total --format=csv

try:
    import torch
except ImportError:
    print("\nTorch belum terpasang. Jalankan sel Install dulu, lalu ulangi sel ini.")
    raise SystemExit("torch belum terpasang")

print("\ntorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("TIDAK ADA GPU. Di Colab: Runtime -> Change runtime type -> T4 GPU, lalu jalankan ulang.")

## 2. Install pustaka latihan (sekali saja per sesi Colab)

In [ ]:
%pip install -q transformers peft trl datasets accelerate
print("Install selesai.")
# Kalau Colab meminta restart runtime setelah install, klik
# "Restart runtime" lalu jalankan lagi dari sel Cek GPU.
!pip uninstall -y torchao  # torchao tidak dibutuhkan untuk LoRA biasa; versi lama (0.10.0) justru membuat peft error


**Penting:** kalau cell install di atas baru saja menghapus torchao, lakukan **Runtime → Restart session** dulu, baru **Runtime → Run all** lagi. Restart memastikan peft tidak lagi melihat torchao lama.

## 3. Upload dataset — pilih file `dataset.jsonl` saat jendela upload muncul

In [ ]:
import json
from google.colab import files

print("Pilih file dataset.jsonl saat jendela upload muncul...")
uploaded = files.upload()
if "dataset.jsonl" not in uploaded:
    raise ValueError(f"File dataset.jsonl tidak ditemukan. Yang ter-upload: {list(uploaded.keys())}. Upload ulang file dataset.jsonl.")

with open("dataset.jsonl", "wb") as f:
    f.write(uploaded["dataset.jsonl"])

rows = [json.loads(line) for line in open("dataset.jsonl", encoding="utf-8") if line.strip()]
print(f"\nJumlah contoh: {len(rows)} (harusnya 37)")
assert all(set(r.keys()) == {"instruction", "input", "output"} for r in rows), "Format baris bukan instruction/input/output!"
print("Satu contoh:")
print("  instruction:", rows[0]["instruction"])
print("  input      :", rows[0]["input"])
print("  output     :", rows[0]["output"][:200], "...")

## 4. Format data — tiap baris jadi teks ChatML (format yang dipakai Qwen)

In [ ]:
# Catatan: training/train_lora.py di repo masih kerangka kosong
# (belum ada fungsi format), jadi format di bawah ini yang dipakai.
SYSTEM = "Kamu adalah Kumar, asisten AI keamanan siber dan coding milik Lion. Jawablah dengan singkat dan benar."

def format_row(r):
    user = r["instruction"].strip()
    if r["input"].strip():
        user += "\n" + r["input"].strip()
    return (
        "<|im_start|>system\n" + SYSTEM + "<|im_end|>\n"
        "<|im_start|>user\n" + user + "<|im_end|>\n"
        "<|im_start|>assistant\n" + r["output"].strip() + "<|im_end|>"
    )

texts = [format_row(r) for r in rows]
print("Contoh teks latih (baris pertama):")
print(texts[0])
print("\nPanjang karakter terpanjang:", max(len(t) for t in texts), "(max_seq_length 512 token cukup)")

## 5. Latih LoRA — Qwen2.5-Coder-0.5B (model terkecil, muat di GPU gratis Colab)

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model

BASE = "Qwen/Qwen2.5-Coder-0.5B"
OUT = "/content/kumar-lora-trial"
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False

def tokenize(text):
    # Loss hanya dihitung pada jawaban (assistant), prompt di-mask dengan -100.
    # Semua contoh di-pad ke MAX_LEN di sini, supaya batch selalu sama panjang:
    # collator di transformers terbaru tidak lagi mem-pad labels, jadi kalau
    # panjang contoh beda-beda (mis. 132 vs 191 token) batch-nya gagal dibentuk.
    prompt = text.split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    full_ids = tokenizer(text, truncation=True, max_length=MAX_LEN)["input_ids"]
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
    n = min(len(prompt_ids), len(full_ids))
    labels = ([-100] * n + full_ids[n:])[:MAX_LEN]
    pad_len = MAX_LEN - len(full_ids)
    input_ids = full_ids + [tokenizer.pad_token_id] * pad_len
    labels = labels + [-100] * pad_len
    attention_mask = [1] * len(full_ids) + [0] * pad_len
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

ds = Dataset.from_list([{"text": t} for t in texts])
ds = ds.map(lambda ex: tokenize(ex["text"]), remove_columns=["text"])

peft_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=OUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print(f"\nSelesai. Adapter tersimpan di {OUT}")

## 6. Uji adapter — muat ulang base + adapter, lalu jawab 3 prompt

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-Coder-0.5B"
OUT = "/content/kumar-lora-trial"

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base, OUT)
model.eval()

SYSTEM = "Kamu adalah Kumar, asisten AI keamanan siber dan coding milik Lion. Jawablah dengan singkat dan benar."

def ask(instruction, inp=""):
    user = instruction.strip() + (("\n" + inp.strip()) if inp.strip() else "")
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=256, do_sample=False,
                         repetition_penalty=1.15,  # cegah jawaban ngulang-ngulang (loop)
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

tests = [
    # ADA di data latih (baris 1) — harusnya dihafal (tes hafalan)
    ("Tulis kode login PHP yang aman dari SQL injection.",
     "Bahasa: PHP, database MySQL, tabel users (username, password_hash)."),
    # ADA di data latih (baris 34) — harusnya dihafal
    ("Cek peringkat entry make.fun-ku di leaderboard kompetisi.",
     "Kompetisi food-picture-01, submission id 01a1d7b5-a669-73f8-8f3f-3b71817d2817."),
    # TIDAK ADA di data latih — ini tes sebenarnya (generalisasi)
    ("Tulis fungsi Python untuk memvalidasi email pengguna.",
     "Pakai regex, tolak email tanpa @ dan tanpa domain."),
]

for i, (ins, inp) in enumerate(tests, 1):
    print(f"===== Uji {i}: {ins}")
    print(ask(ins, inp))
    print()

## 7. (Opsional) Unduh adapter hasil latihan

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/kumar-lora-trial", "zip", "/content/kumar-lora-trial")
print("Adapter di-zip. Klik download saat jendela muncul...")
files.download("/content/kumar-lora-trial.zip")

## Langkah selanjutnya

- Kalau semua sel jalan dan uji 1–2 keluar mirip contoh latihnya, **jalur latihan terbukti**. Uji 3 yang tidak ada di data latih kemungkinan masih ngawur — itu normal dengan 37 contoh.
- Terus kumpulkan ilmu (lab pagi, bounty, contoh keamanan baru) sampai dataset mencapai **500+ contoh**, lalu latih ulang dengan notebook yang sama — cukup ganti file `dataset.jsonl` yang di-upload.
- Adapter hasil latihan ini (`kumar-lora-trial`) jangan dipakai untuk memperbaiki website dulu — tunggu model hasil latih dengan data yang jauh lebih banyak dan sudah dievaluasi.